# TP 2 (avancé) — Qualité et préparation des données
**Analyse de données — Master IA / Master RSD** · Chapitres 1–2 : exploration et préparation des données

Un e-commerce algérien vous confie deux extractions **brutes** de son système :
- `clients_bruts.csv` — un fichier **structuré** mais sale (doublons, types erronés, valeurs impossibles, catégories incohérentes…) ;
- `achats.json` — un fichier **semi-structuré** : une liste d'achats imbriquée dans chaque client.

**Mission :** produire un jeu de données **propre, enrichi et prêt pour la modélisation**, sans fuite de données.

| Partie | Thème | Notions |
|---|---|---|
| 1 | Import multi-sources | `read_csv`, `json_normalize` |
| 2 | Audit qualité | doublons, types, manquants, modalités |
| 3 | Nettoyage | conversion de types, normalisation de texte, dates |
| 4 | Valeurs aberrantes | IQR, z-score, codes sentinelles, transformation log |
| 5 | Valeurs manquantes | MCAR/MAR/MNAR, imputation par groupe, indicateur |
| 6 | Ingénierie de variables | ancienneté, RFM, `merge`, `pd.cut` |
| 7 | Exploration | corrélations, `pivot_table`, `crosstab` |
| 8 | Préparation à la modélisation | `train_test_split`, `ColumnTransformer`, fuite de données |
| 9 | Défi | une fonction `nettoyer()` reproductible |

> Travail en binôme. Complétez les cellules **`# TODO`**. Les cellules **✅ Auto-correction** valident vos résultats : si une erreur s'affiche, corrigez la cellule précédente avant de continuer — chaque partie dépend de la précédente.

---
## Mise en place

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
DATE_REF = pd.Timestamp("2026-01-01")      # date de référence de l'analyse

---
## Partie 1 — Import multi-sources
### 1.1 Le fichier structuré
Importez `clients_bruts.csv` dans `df_brut` (séparateur `;`). **Ne modifiez jamais `df_brut`** : on travaillera sur une copie, pour pouvoir toujours revenir à la source.

In [ ]:
df_brut = ...   # TODO

df_brut.head()

✅ **Auto-correction**

In [ ]:
assert df_brut.shape == (1025, 8), f"Attendu (1025, 8), obtenu {df_brut.shape} -> séparateur ?"
print("✅ 1.1 correct")

### 1.2 Le fichier semi-structuré
`achats.json` est une **liste de clients**, chacun portant une **liste d'achats**. Regardez d'abord sa forme brute, puis « aplatissez-le » en un tableau d'**une ligne par achat** avec `pd.json_normalize(data, record_path=..., meta=...)`.

In [ ]:
with open("achats.json", encoding="utf-8") as f:
    data = json.load(f)

data[0]

In [ ]:
achats = ...   # TODO : json_normalize, une ligne par achat, avec la colonne client_id

achats.head()

✅ **Auto-correction**

In [ ]:
assert list(achats.columns) == ["date", "montant", "categorie", "client_id"], "Colonnes attendues : date, montant, categorie, client_id"
assert len(achats) == 5160, "Nombre d'achats incorrect"
assert achats["client_id"].nunique() == 900, "900 clients ont au moins un achat"
print("✅ 1.2 correct")

---
## Partie 2 — Audit qualité
### 2.1 Une fonction d'audit réutilisable
Écrivez `audit(df)` qui renvoie un DataFrame avec **une ligne par colonne** de `df` et les colonnes :
`type` (dtype), `manquants` (nombre de NaN), `pct_manquants` (en %, arrondi à 0,1), `n_uniques` (nombre de valeurs distinctes).

In [ ]:
def audit(df):
    # TODO
    ...

audit(df_brut)

✅ **Auto-correction**

In [ ]:
a = audit(df_brut)
assert list(a.columns) == ["type", "manquants", "pct_manquants", "n_uniques"]
assert a.loc["revenu", "type"] == "object", "revenu est lu comme du texte : c'est normal à ce stade"
assert a.loc["ville", "manquants"] == 12
print("✅ 2.1 correct")

### 2.2 Lire l'audit
1. Combien de lignes sont des **doublons exacts** ? Combien de `client_id` distincts ?
2. Pourquoi `revenu` est-il de type `object` ? Affichez quelques valeurs.
3. Combien de **modalités** distinctes pour `ville` et `satisfaction` ? Combien devrait-il y en avoir ?
4. Quelles valeurs d'`age` sont impossibles ?

In [ ]:
n_doublons = ...   # TODO
print(n_doublons, df_brut["client_id"].nunique())

# TODO : quelques valeurs de revenu, value_counts de ville et satisfaction, describe de age


✅ **Auto-correction**

In [ ]:
assert n_doublons == 25, "25 lignes dupliquées attendues"
print("✅ 2.2 correct")

✍️ **Vos constats :** …

---
## Partie 3 — Nettoyage
Travaillez sur `df = df_brut.copy()`. Effectuez dans l'ordre :
1. supprimer les **doublons** et réinitialiser l'index ;
2. convertir `revenu` en **numérique** (retirer les espaces, puis `pd.to_numeric(..., errors="coerce")`) ;
3. normaliser `ville` : sans espaces autour, **Première lettre en majuscule** (`str.strip().str.title()`) ;
4. normaliser `satisfaction` : sans espaces, en minuscules, et `"elevee"` → `"élevée"` ;
5. convertir `date_inscription` en `datetime` (deux formats mélangés : `format="mixed", dayfirst=True`) ;
6. remplacer les âges hors de **[18, 100]** par `NaN`.

In [ ]:
df = df_brut.copy()

# 1. doublons
# TODO

# 2. revenu -> numérique
# TODO

# 3. ville
# TODO

# 4. satisfaction
# TODO

# 5. date_inscription
# TODO

# 6. âges impossibles
# TODO

audit(df)

✅ **Auto-correction**

In [ ]:
assert df.shape == (1000, 8), "1000 lignes après suppression des doublons"
assert df["revenu"].dtype == "float64", "revenu doit être numérique"
assert df["revenu"].isna().sum() == 40, "40 revenus vides attendus"
assert sorted(df["ville"].dropna().unique()) == ["Alger", "Annaba", "Constantine", "Oran", "Tlemcen"]
assert sorted(df["satisfaction"].unique()) == ["faible", "moyenne", "élevée"]
assert pd.api.types.is_datetime64_any_dtype(df["date_inscription"]) and df["date_inscription"].notna().all()
assert df["age"].isna().sum() == 8 and df["age"].between(18, 100).sum() == 992
assert df_brut.shape == (1025, 8), "df_brut ne doit pas avoir été modifié !"
print("✅ Partie 3 correcte")

---
## Partie 4 — Valeurs aberrantes
### 4.1 Deux règles de détection
Sur `revenu` :
- **règle IQR** : aberrant si $x < Q_1 - 1{,}5\,IQR$ ou $x > Q_3 + 1{,}5\,IQR$ ;
- **règle z-score** : aberrant si $|z| > 3$ avec $z = (x - \bar x)/\sigma$.

Comptez les revenus signalés par chaque règle.

In [ ]:
q1, q3 = ...   # TODO
iqr = q3 - q1
aberrant_iqr = ...   # TODO : Series booléenne
z = ...              # TODO
aberrant_z = ...     # TODO

print("IQR :", aberrant_iqr.sum(), "| z-score :", aberrant_z.sum())

✅ **Auto-correction**

In [ ]:
assert aberrant_iqr.sum() == 42 and aberrant_z.sum() == 9
print("✅ 4.1 correct")

In [ ]:
df.loc[aberrant_iqr, "revenu"].sort_values(ascending=False).value_counts(sort=False).head(15)

### 4.2 Aberrant ≠ erroné
Les deux règles ne disent pas la même chose, et **aucune** ne distingue une erreur d'un vrai cas extrême.
1. Une valeur revient **exactement** plusieurs fois tout en haut : laquelle ? Que représente-t-elle probablement ?
2. Remplacez ce **code sentinelle** par `NaN` (c'est une valeur manquante déguisée), mais **gardez** les vrais hauts revenus.

In [ ]:
n_codes = ...   # TODO : nombre de revenus égaux au code
# TODO : remplacer le code par NaN


✅ **Auto-correction**

In [ ]:
assert n_codes == 3 and df["revenu"].isna().sum() == 43 and df["revenu"].max() < 999999
print("✅ 4.2 correct")

### 4.3 Asymétrie et transformation logarithmique
Le revenu est **asymétrique à droite** : la règle IQR signale alors de nombreux clients parfaitement normaux. Créez `revenu_log = np.log1p(revenu)`, comparez l'**asymétrie** (`.skew()`) avant/après, tracez les deux histogrammes, puis recomptez les aberrants IQR **sur l'échelle log**.

In [ ]:
# TODO : df["revenu_log"] = ...
# TODO : skew avant / après
# TODO : deux histogrammes côte à côte
# TODO : aberrants IQR sur revenu_log


**Point de sauvegarde** : `df_propre` est le résultat du nettoyage (Parties 3–4). On le réutilisera au défi final.

In [ ]:
df_propre = df.copy()

---
## Partie 5 — Valeurs manquantes
### 5.1 Le manque est-il aléatoire ?
On distingue :
- **MCAR** (*Missing Completely At Random*) : le manque ne dépend de rien ;
- **MAR** (*At Random*) : il dépend d'**autres** variables observées (ex. le segment) ;
- **MNAR** (*Not At Random*) : il dépend de la valeur **elle-même** (ex. les hauts revenus refusent de répondre).

Calculez le **taux de revenus manquants par segment**. Le manque semble-t-il lié au segment ?

In [ ]:
# TODO : taux de NaN de revenu par segment


### 5.2 Imputer (pour l'analyse descriptive)
Sur une **copie** `df_desc` :
1. ajoutez un indicateur `revenu_manquant` (0/1) **avant** d'imputer — l'absence d'information est une information ;
2. imputez `revenu` par la **médiane de son segment** (`groupby(...).transform("median")`) ;
3. comparez avec une imputation par la **médiane globale** : quelle méthode préserve mieux les écarts entre segments ?

In [ ]:
df_desc = df.copy()
# TODO


✅ **Auto-correction**

In [ ]:
assert df_desc["revenu"].isna().sum() == 0 and df_desc["revenu_manquant"].sum() == 43
assert df["revenu"].isna().sum() == 43, "df ne doit pas être imputé (travaillez sur df_desc)"
print("✅ 5.2 correct")

⚠️ **Question piège** : si l'objectif est plus tard de **prédire le segment**, peut-on utiliser cette imputation par segment dans le modèle ? Répondez avant la Partie 8.

✍️ …

---
## Partie 6 — Ingénierie de variables
### 6.1 Variables dérivées des clients
- `anciennete_ans` : années écoulées entre `date_inscription` et `DATE_REF` (arrondi à 0,1) ;
- `tranche_age` : `pd.cut` de l'âge en `[18-25[`, `[25-35[`, `[35-50[`, `[50-65[`, `65+` (bornes à gauche incluses : `right=False`).

In [ ]:
# TODO : df["anciennete_ans"] = ...
# TODO : df["tranche_age"] = pd.cut(...)


✅ **Auto-correction**

In [ ]:
assert df["anciennete_ans"].between(0, 11.1).all()
assert df["tranche_age"].isna().sum() == 8, "seuls les 8 âges inconnus restent sans tranche"
print("✅ 6.1 correct")

### 6.2 Indicateurs RFM à partir des achats
Le **RFM** est un classique du marketing :
- **R**écence : nombre de jours entre le **dernier** achat et `DATE_REF` ;
- **F**réquence : nombre d'achats ;
- **M**ontant : somme dépensée.

1. Convertissez `achats["date"]` en datetime, calculez `rfm` (une ligne par client) avec `groupby(...).agg(...)`.
2. Fusionnez dans `df` avec `merge(..., how="left", validate="one_to_one")`.
3. Les 100 clients **sans achat** ont des NaN : quelle valeur est logique pour `frequence` et `montant_total` ? Et pour `recence` ?

In [ ]:
# TODO


✅ **Auto-correction**

In [ ]:
assert len(df) == 1000, "le merge ne doit ni perdre ni dupliquer de clients"
assert df["frequence"].sum() == len(achats) and (df["frequence"] == 0).sum() == 100
assert df["montant_total"].sum() == achats["montant"].sum()
print("✅ 6.2 correct")

---
## Partie 7 — Exploration
1. Tracez la **matrice de corrélation** des variables numériques (`sns.heatmap`, `annot=True`).
2. Tableau croisé du **montant moyen** par `segment` × `niveau_etude` (`pivot_table`).
3. Répartition de la `satisfaction` **en %** au sein de chaque segment (`pd.crosstab(..., normalize="index")`).

Rédigez **trois constats** chiffrés.

In [ ]:
# TODO


✍️ **Vos trois constats :**
1. …
2. …
3. …

---
## Partie 8 — Préparer les données pour un modèle
**Objectif** : préparer `X` (variables explicatives) et `y = segment` (cible) pour un futur modèle de classification.

Règle d'or : **tout ce qui s'apprend sur les données** (médiane d'imputation, moyenne/écart-type du scaling, modalités du one-hot) **s'apprend sur le jeu d'entraînement seul**, puis s'applique au test. Sinon, le test « fuit » dans l'entraînement et la performance mesurée est trop optimiste.

1. Séparez `X` / `y`, puis `train_test_split` (20 % de test, `stratify=y`, `random_state=0`).
2. Construisez un `ColumnTransformer` :
   - **numériques** `age, revenu_log, anciennete_ans, recence, frequence, montant_total` → `SimpleImputer(median)` + `StandardScaler` ;
   - **ordinales** `niveau_etude, satisfaction` → `SimpleImputer(most_frequent)` + `OrdinalEncoder` **avec l'ordre explicite** ;
   - **nominale** `ville` → `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown="ignore")`.
3. `fit_transform` sur le train, `transform` sur le test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

col_num = ["age", "revenu_log", "anciennete_ans", "recence", "frequence", "montant_total"]
col_ord = ["niveau_etude", "satisfaction"]
col_nom = ["ville"]

In [ ]:
X = ...   # TODO
y = ...   # TODO
X_train, X_test, y_train, y_test = ...   # TODO

prep = ColumnTransformer([
    # TODO : ("num", ..., col_num), ("ord", ..., col_ord), ("nom", ..., col_nom)
])

X_train_p = ...   # TODO
X_test_p  = ...   # TODO
print(X_train_p.shape, X_test_p.shape)

✅ **Auto-correction**

In [ ]:
assert X_train_p.shape == (800, 13) and X_test_p.shape == (200, 13), "6 numériques + 2 ordinales + 5 villes = 13 colonnes"
assert not np.isnan(X_train_p).any() and not np.isnan(X_test_p).any()
assert abs(X_train_p[:, 0].mean()) < 1e-9, "le scaler doit être appris sur le train"
print("✅ Partie 8 correcte")

In [ ]:
pd.DataFrame(X_train_p, columns=prep.get_feature_names_out()).head()

### 8.2 Bonus — un premier modèle
Chaînez `prep` et une `LogisticRegression` dans un même `Pipeline`, entraînez sur le train et donnez l'**exactitude** sur le test. (Les modèles seront étudiés plus tard : l'objectif est de voir que la préparation s'intègre au modèle.)

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO


---
## Partie 9 — Défi : un nettoyage reproductible
Dans un vrai projet, les données brutes **reviennent chaque mois**. Regroupez les Parties 3 et 4 dans une fonction `nettoyer(df_brut)` qui renvoie un nouveau DataFrame **identique à `df_propre`**, sans modifier son argument. Puis exportez le résultat en `clients_propres.csv`.

In [ ]:
def nettoyer(brut):
    # TODO
    ...

✅ **Auto-correction**

In [ ]:
copie = df_brut.copy()
pd.testing.assert_frame_equal(nettoyer(df_brut), df_propre)
pd.testing.assert_frame_equal(df_brut, copie)   # argument non modifié
print("✅ Défi réussi : nettoyage reproductible")

In [ ]:
nettoyer(df_brut).to_csv("clients_propres.csv", sep=";", index=False, encoding="utf-8")

---
## Bilan
| Problème | Détection | Traitement |
|---|---|---|
| Doublons | `duplicated()` | `drop_duplicates()` |
| Nombre stocké en texte | `dtypes` = `object` | `str.replace` + `to_numeric(errors="coerce")` |
| Modalités incohérentes | `value_counts()` | `str.strip().str.lower()/.title()` + `replace` |
| Dates en formats mélangés | inspection | `to_datetime(format="mixed", dayfirst=True)` |
| Valeurs impossibles | `describe()`, bornes métier | → `NaN` |
| Codes sentinelles | valeur répétée en extrême | → `NaN` |
| Aberrants statistiques | IQR (sur log), z-score | **investiguer**, garder si plausibles |
| Manquants | `isna()`, taux par groupe | indicateur + imputation **dans le pipeline** |
| Fuite de données | — | `fit` sur le train uniquement, jamais la cible |

> **À retenir :** nettoyer, c'est **décider** — et chaque décision doit être justifiée, tracée et reproductible.